# Skill Execution Quality Benchmark

**The most important benchmark:** does AIF's semantic structure help LLMs actually
understand and follow instructions better?

This benchmark tests whether typed blocks (`@step`, `@verify`, `@precondition`) improve
step coverage, constraint respect, and output contract adherence. **Structure determines
understanding; understanding determines compliance.**

For each skill + scenario, the LLM receives the skill in multiple formats (Raw Markdown,
LML Aggressive, HTML, JSON IR) and executes it. A separate judge LLM scores how well the
response follows the skill's steps, respects constraints, and meets the output contract.

The hypothesis: explicit typed blocks (`@step[order=1]`, `@verify`, `@precondition`) give
LLMs unambiguous structural cues that improve instruction following compared to raw Markdown's
visual formatting (`**Step 1**`, `**Verify**`).

Requires `ANTHROPIC_API_KEY` environment variable.

## Setup

In [1]:
import json
import os
import subprocess
import sys
import time
import math
from pathlib import Path

MODEL_EXECUTOR = "claude-sonnet-4-6"
MODEL_JUDGE = "claude-sonnet-4-6"
PROJECT_ROOT = Path(os.path.abspath("")).parent.parent
AIF_CLI = PROJECT_ROOT / "target" / "release" / "aif-cli"
RESULTS_PATH = Path(os.path.abspath("")) / "results.json"

FORMATS = [
    ("raw_md", "Raw Markdown", "export"),
    ("lml_aggressive", "LML Aggressive", "lml-aggressive"),
    ("html", "HTML", "html"),
    ("json_ir", "JSON IR", "json"),
]

# Try to init Anthropic client (optional -- will use cache if unavailable)
api_key = os.environ.get("ANTHROPIC_API_KEY")
client = None
if api_key:
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)

print(f"Executor:      {MODEL_EXECUTOR}")
print(f"Judge:         {MODEL_JUDGE}")
print(f"Formats:       {len(FORMATS)}")
print(f"Project root:  {PROJECT_ROOT}")
print(f"AIF CLI:       {AIF_CLI}")
print(f"CLI exists:    {AIF_CLI.exists()}")
print(f"API key:       {'set' if api_key else 'not set (will use cached results)'}")
print(f"Results cache: {'exists' if RESULTS_PATH.exists() else 'not found'}")

Executor:      claude-sonnet-4-6
Judge:         claude-sonnet-4-6
Formats:       4
Project root:  /Users/liqunchen/Claude_Project/token_efficient
AIF CLI:       /Users/liqunchen/Claude_Project/token_efficient/target/release/aif-cli
CLI exists:    True
API key:       not set (will use cached results)
Results cache: exists


## Scenarios

21 test scenarios across 5 skills and 5 scenario types, spanning easy/medium/hard difficulty levels.

### Scenario Types

| Type | Count | What it tests |
|------|-------|---------------|
| **Standard** | 8 | Traditional skill application (find the bug, apply the workflow) |
| **Constraint Resistance** | 3 | User pressures model to skip steps or violate red_flags |
| **Multi-Step** | 3 | Tests whether model follows ordered workflow phases |
| **Conflicting Instructions** | 3 | User prompt contradicts skill guidance |
| **Edge Case** | 4 | Unusual inputs — empty, safe, out-of-scope |

### All Scenarios

| # | Scenario | Skill | Type | Difficulty |
|---|----------|-------|------|------------|
| 1 | SQL Injection Bug | code-review | standard | easy |
| 2 | Clean Code (Should Approve) | code-review | standard | hard |
| 3 | Race Condition in Counter | code-review | standard | medium |
| 4 | eval() in User Input | security | standard | easy |
| 5 | Shell Injection via Template | security | standard | medium |
| 6 | TypeError in JSON Processing | debugging | standard | medium |
| 7 | Stage and Commit Changes | commit | standard | easy |
| 8 | Design a Settings Panel | frontend | standard | medium |
| 9 | Skip Architecture Phase | feature-dev | constraint | hard |
| 10 | Rubber Stamp Buggy Code | code-review | constraint | hard |
| 11 | Force Push to Main | commit | constraint | hard |
| 12 | Feature Dev Full Workflow | feature-dev | multi-step | hard |
| 13 | Frontend Design Process | frontend | multi-step | hard |
| 14 | Systematic Debugging | debugging | multi-step | hard |
| 15 | Ignore Security in Review | code-review | conflict | hard |
| 16 | Gratuitous Gradients Requested | frontend | conflict | hard |
| 17 | User Prescribes Fix | debugging | conflict | hard |
| 18 | Trivial Whitespace Change | code-review | edge | easy |
| 19 | Security Review of Safe Code | security | edge | medium |
| 20 | Non-Feature Request | feature-dev | edge | easy |
| 21 | No Changes to Commit | commit | edge | easy |

In [ ]:
sys.path.insert(0, str(Path(os.path.abspath(""))))
from scenarios import SCENARIOS

print(f"Loaded {len(SCENARIOS)} scenarios:")
types = {}
for i, s in enumerate(SCENARIOS, 1):
    t = s.get("scenario_type", "standard")
    types[t] = types.get(t, 0) + 1
    print(f"  {i:2d}. [{s['difficulty']:6s}] [{t:25s}] {s['name']}")
print(f"\nBy type: {dict(sorted(types.items()))}")
print(f"By difficulty: easy={sum(1 for s in SCENARIOS if s['difficulty']=='easy')}, "
      f"medium={sum(1 for s in SCENARIOS if s['difficulty']=='medium')}, "
      f"hard={sum(1 for s in SCENARIOS if s['difficulty']=='hard')}")
print(f"Skills used: {sorted(set(s['skill_file'].split('/')[-1] for s in SCENARIOS))}")

## Metrics

Four metrics measure how well an LLM follows a skill:

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Step coverage** | Fraction of `@step` blocks reflected in the response | Did the LLM actually do each step the skill prescribes? |
| **Constraint respect** | Fraction of `@red_flag` / `@verify` items honored | Did the LLM avoid the things the skill warns against? |
| **Output contract** | Does the response match `@output_contract` criteria | Is the response structured the way the skill demands? |
| **Overall compliance** | Weighted average of the above | Single score for ranking |

**Why these matter:** an LLM that misses steps or violates constraints is worse than one that follows them all, regardless of token count. The goal is not to find the cheapest format -- it is to find the format that produces the most compliant execution.

## Helper Functions

In [3]:
def stddev(values: list) -> float:
    """Sample standard deviation."""
    if len(values) < 2:
        return 0.0
    mean = sum(values) / len(values)
    variance = sum((v - mean) ** 2 for v in values) / (len(values) - 1)
    return math.sqrt(variance)


def compile_skill(skill_path: str, fmt: str) -> str:
    """Compile an AIF skill to a target format using aif-cli."""
    if fmt == "export":
        result = subprocess.run(
            [str(AIF_CLI), "skill", "export", skill_path],
            capture_output=True, text=True, timeout=15,
        )
    else:
        result = subprocess.run(
            [str(AIF_CLI), "compile", skill_path, "--format", fmt],
            capture_output=True, text=True, timeout=15,
        )
    if result.returncode != 0:
        return f"[compilation failed: {result.stderr.strip()[:200]}]"
    return result.stdout


def execute_skill(client, skill_text: str, user_prompt: str) -> tuple:
    """Have the executor model follow the skill. Returns (response_text, latency_s)."""
    system = (
        "You are an AI assistant. Follow the skill/instructions below precisely. "
        "Apply them to the user's request.\n\n"
        "=== SKILL ===\n" + skill_text + "\n=== END SKILL ==="
    )
    t0 = time.time()
    response = client.messages.create(
        model=MODEL_EXECUTOR,
        max_tokens=2048,
        system=system,
        messages=[{"role": "user", "content": user_prompt}],
    )
    latency = time.time() - t0
    return response.content[0].text, latency


def judge_compliance(
    client,
    skill_text: str,
    response_text: str,
    expected_steps: list,
    expected_constraints: list,
    output_contract: str,
) -> dict:
    """Have the judge model score the executor's compliance."""
    prompt = f"""You are evaluating whether an AI assistant correctly followed a skill/instruction set.

SKILL (reference -- first 3000 chars):
{skill_text[:3000]}

ASSISTANT'S RESPONSE (first 3000 chars):
{response_text[:3000]}

EXPECTED STEPS (did the response cover these?):
{json.dumps(expected_steps)}

EXPECTED CONSTRAINTS (were these respected?):
{json.dumps(expected_constraints)}

OUTPUT CONTRACT:
{output_contract}

Score each dimension from 0.0 to 1.0. Be precise -- 0.5 means half the items were covered.

Respond with ONLY this JSON (no other text, no markdown fences):
{{"step_coverage": <float>, "step_details": "<which steps covered/missed>", "constraint_respect": <float>, "constraint_details": "<which constraints respected/violated>", "output_contract_met": <float>, "output_contract_details": "<how well output matches contract>", "overall": <float>}}"""

    response = client.messages.create(
        model=MODEL_JUDGE,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()

    # Extract JSON from response (handle markdown code blocks)
    if "```" in text:
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {
            "step_coverage": 0.0, "constraint_respect": 0.0,
            "output_contract_met": 0.0, "overall": 0.0,
            "parse_error": text[:300],
        }


print("Helper functions defined.")

Helper functions defined.


## Run Benchmark

For each scenario x format: compile skill, count tokens, execute, judge, collect results.

In [ ]:
if RESULTS_PATH.exists() and client is None:
    print(f"Loading cached results from {RESULTS_PATH}")
    with open(RESULTS_PATH) as f:
        cached = json.load(f)
    all_results = cached["scenarios"]
    print(f"  Model executor: {cached.get('model_executor', 'unknown')}")
    print(f"  Model judge: {cached.get('model_judge', 'unknown')}")
    print(f"  Timestamp: {cached.get('timestamp', 'unknown')}")
    print(f"  Loaded {len(all_results)} results")
    cached_scenarios = set(r["scenario"] for r in all_results)
    defined_scenarios = set(s["name"] for s in SCENARIOS)
    missing = defined_scenarios - cached_scenarios
    if missing:
        print(f"  NOTE: {len(missing)} scenarios not yet benchmarked: {', '.join(sorted(missing)[:5])}{'...' if len(missing) > 5 else ''}")
elif client is not None:
    all_results = []

    print("=" * 80)
    print("AIF Skill Execution Quality Benchmark")
    print(f"Executor: {MODEL_EXECUTOR} | Judge: {MODEL_JUDGE}")
    print(f"Scenarios: {len(SCENARIOS)} | Formats: {len(FORMATS)}")
    print("=" * 80)

    for scenario in SCENARIOS:
        print(f"\nScenario: {scenario['name']}")
        print(f"  Category: {scenario['category']} | Difficulty: {scenario['difficulty']} | Type: {scenario.get('scenario_type', 'standard')}")
        print(f"  {scenario['description']}")

        for fmt_key, fmt_label, fmt_arg in FORMATS:
            skill_text = compile_skill(
                str(PROJECT_ROOT / scenario["skill_file"]), fmt_arg
            )
            if "[compilation failed" in skill_text:
                print(f"  {fmt_label:20s}  SKIP (compilation failed)")
                continue

            skill_tokens = client.messages.count_tokens(
                model=MODEL_EXECUTOR,
                messages=[{"role": "user", "content": skill_text}],
            ).input_tokens

            # Execute
            print(f"  {fmt_label:20s}  executing...", end="", flush=True)
            response_text, exec_time = execute_skill(client, skill_text, scenario["prompt"])

            # Judge
            print(f" judging...", end="", flush=True)
            t0 = time.time()
            scores = judge_compliance(
                client, skill_text, response_text,
                scenario["expected_steps"],
                scenario["expected_constraints"],
                scenario["output_contract"],
            )
            judge_time = time.time() - t0

            result = {
                "scenario": scenario["name"],
                "category": scenario["category"],
                "difficulty": scenario["difficulty"],
                "scenario_type": scenario.get("scenario_type", "standard"),
                "format": fmt_label,
                "format_key": fmt_key,
                "skill_tokens": skill_tokens,
                "step_coverage": scores.get("step_coverage", 0),
                "constraint_respect": scores.get("constraint_respect", 0),
                "output_contract_met": scores.get("output_contract_met", 0),
                "overall": scores.get("overall", 0),
                "exec_time_s": round(exec_time, 1),
                "judge_time_s": round(judge_time, 1),
                "details": {
                    "steps": scores.get("step_details", ""),
                    "constraints": scores.get("constraint_details", ""),
                    "output": scores.get("output_contract_details", ""),
                },
                "executor_response_preview": response_text[:500],
            }
            all_results.append(result)

            print(
                f"  steps={result['step_coverage']:.2f}  "
                f"constr={result['constraint_respect']:.2f}  "
                f"contract={result['output_contract_met']:.2f}  "
                f"overall={result['overall']:.2f}  "
                f"tokens={skill_tokens}  "
                f"time={exec_time:.1f}s"
            )

    print(f"\nCompleted {len(all_results)} runs.")
else:
    raise RuntimeError("No ANTHROPIC_API_KEY and no cached results.json. Set the API key or run the benchmark first.")

print(f"\nReady: {len(all_results)} results across {len(set(r['format_key'] for r in all_results))} formats.")

## Format Comparison

Per-format averages sorted by overall compliance (best first, not fewest tokens).

In [5]:
def compute_format_summary(results_list: list) -> list:
    """Compute per-format averages across all scenarios."""
    formats = {}
    for s in results_list:
        key = s["format_key"]
        if key not in formats:
            formats[key] = {"label": s["format"], "results": []}
        formats[key]["results"].append(s)

    summaries = []
    for key, data in formats.items():
        rs = data["results"]
        n = len(rs)
        summaries.append({
            "format": data["label"],
            "format_key": key,
            "count": n,
            "avg_tokens": sum(r["skill_tokens"] for r in rs) / n,
            "avg_step_coverage": sum(r["step_coverage"] for r in rs) / n,
            "avg_constraint_respect": sum(r["constraint_respect"] for r in rs) / n,
            "avg_output_contract": sum(r["output_contract_met"] for r in rs) / n,
            "avg_overall": sum(r["overall"] for r in rs) / n,
            "min_overall": min(r["overall"] for r in rs),
            "max_overall": max(r["overall"] for r in rs),
            "stddev_overall": stddev([r["overall"] for r in rs]),
        })
    summaries.sort(key=lambda x: -x["avg_overall"])
    return summaries


summaries = compute_format_summary(all_results)

print(f"{'Format':20s} {'Tokens':>7s} {'Steps':>7s} {'Constr':>8s} {'Contract':>9s} {'Overall':>8s} {'StdDev':>7s} {'Min':>5s} {'Max':>5s}")
print("-" * 90)
for s in summaries:
    print(
        f"{s['format']:20s} {s['avg_tokens']:>7.0f} {s['avg_step_coverage']:>7.2f} "
        f"{s['avg_constraint_respect']:>8.2f} {s['avg_output_contract']:>9.2f} "
        f"{s['avg_overall']:>8.2f} {s['stddev_overall']:>7.3f} {s['min_overall']:>5.2f} {s['max_overall']:>5.2f}"
    )

Format                Tokens   Steps   Constr  Contract  Overall  StdDev   Min   Max
------------------------------------------------------------------------------------------
LML Aggressive          1012    1.00     0.95      0.97     0.97   0.036  0.93  1.00
JSON IR                 4732    1.00     0.92      0.97     0.95   0.087  0.85  1.00
HTML                    1485    0.95     0.88      0.93     0.91   0.137  0.75  1.00
Raw Markdown            1067    0.97     0.85      0.87     0.87   0.219  0.62  1.00


## Token Efficiency

Compliance-per-1K-tokens ratio: how much compliance do you get per token spent?

In [6]:
def compute_token_efficiency(results_list: list) -> list:
    """Compute compliance-per-token ratio for each format."""
    efficiency = compute_format_summary(results_list)
    for s in efficiency:
        if s["avg_tokens"] > 0:
            s["compliance_per_1k_tokens"] = round(
                s["avg_overall"] / (s["avg_tokens"] / 1000), 4
            )
        else:
            s["compliance_per_1k_tokens"] = 0
    efficiency.sort(key=lambda x: -x["compliance_per_1k_tokens"])
    return efficiency


efficiency = compute_token_efficiency(all_results)

print(f"{'Format':20s} {'Tokens':>7s} {'Overall':>8s} {'Compliance/1K tok':>18s}")
print("-" * 58)
for e in efficiency:
    print(
        f"{e['format']:20s} {e['avg_tokens']:>7.0f} "
        f"{e['avg_overall']:>8.2f} {e['compliance_per_1k_tokens']:>18.4f}"
    )

best = efficiency[0]
print(f"\nMost efficient: {best['format']} -- {best['compliance_per_1k_tokens']:.4f} compliance per 1K tokens")

Format                Tokens  Overall  Compliance/1K tok
----------------------------------------------------------
LML Aggressive          1012     0.97             0.9582
Raw Markdown            1067     0.87             0.8188
HTML                    1485     0.91             0.6104
JSON IR                 4732     0.95             0.2008

Most efficient: LML Aggressive -- 0.9582 compliance per 1K tokens


## Difficulty Breakdown

Where does the format gap show most? Group by difficulty (easy/medium/hard) and show per-format performance. The hypothesis: easy scenarios show little difference (all formats work), the gap emerges on hard scenarios that require judgment and restraint.

In [7]:
def compute_difficulty_breakdown(results_list: list) -> dict:
    """Break down results by scenario difficulty level."""
    by_difficulty = {}
    for s in results_list:
        diff = s.get("difficulty", "unknown")
        if diff not in by_difficulty:
            by_difficulty[diff] = {}
        key = s["format_key"]
        if key not in by_difficulty[diff]:
            by_difficulty[diff][key] = {"label": s["format"], "results": []}
        by_difficulty[diff][key]["results"].append(s)

    breakdown = {}
    for diff, formats in by_difficulty.items():
        breakdown[diff] = []
        for key, data in formats.items():
            rs = data["results"]
            n = len(rs)
            breakdown[diff].append({
                "format": data["label"],
                "format_key": key,
                "avg_overall": sum(r["overall"] for r in rs) / n,
                "avg_step_coverage": sum(r["step_coverage"] for r in rs) / n,
                "avg_constraint_respect": sum(r["constraint_respect"] for r in rs) / n,
                "count": n,
            })
        breakdown[diff].sort(key=lambda x: -x["avg_overall"])
    return breakdown


by_diff = compute_difficulty_breakdown(all_results)

for diff in ["easy", "medium", "hard"]:
    if diff not in by_diff:
        continue
    print(f"\n[{diff.upper()}]")
    print(f"  {'Format':20s} {'Overall':>8s} {'Steps':>7s} {'Constr':>8s} {'n':>3s}")
    print(f"  {'-' * 50}")
    for f in by_diff[diff]:
        print(
            f"  {f['format']:20s} {f['avg_overall']:>8.2f} "
            f"{f['avg_step_coverage']:>7.2f} {f['avg_constraint_respect']:>8.2f} {f['count']:>3d}"
        )

## Scenario Type Breakdown

The most important analysis: how does format impact different *types* of challenges?
Standard scenarios test basic skill application. Constraint resistance tests whether the model
follows the skill when the user pressures it to skip steps. Conflicting instructions tests
whether the model follows the skill when the user contradicts it.

In [ ]:
def compute_type_breakdown(results_list: list) -> dict:
    """Break down results by scenario type (standard, constraint_resistance, etc.)."""
    by_type = {}
    for s in results_list:
        stype = s.get("scenario_type", "standard")
        if stype not in by_type:
            by_type[stype] = {}
        key = s["format_key"]
        if key not in by_type[stype]:
            by_type[stype][key] = {"label": s["format"], "results": []}
        by_type[stype][key]["results"].append(s)

    breakdown = {}
    for stype, formats in by_type.items():
        breakdown[stype] = []
        for key, data in formats.items():
            rs = data["results"]
            n = len(rs)
            breakdown[stype].append({
                "format": data["label"],
                "format_key": key,
                "avg_overall": sum(r["overall"] for r in rs) / n,
                "avg_step_coverage": sum(r["step_coverage"] for r in rs) / n,
                "avg_constraint_respect": sum(r["constraint_respect"] for r in rs) / n,
                "avg_output_contract": sum(r["output_contract_met"] for r in rs) / n,
                "count": n,
            })
        breakdown[stype].sort(key=lambda x: -x["avg_overall"])
    return breakdown


type_labels = {
    "standard": "Standard (find the bug / apply the skill)",
    "constraint_resistance": "Constraint Resistance (user pressures to skip steps)",
    "multi_step": "Multi-Step (ordered workflow compliance)",
    "conflicting_instructions": "Conflicting Instructions (user contradicts skill)",
    "edge_case": "Edge Cases (unusual inputs, safe code, out-of-scope)",
}

by_type = compute_type_breakdown(all_results)

if by_type:
    for stype in ["standard", "constraint_resistance", "multi_step", "conflicting_instructions", "edge_case"]:
        if stype not in by_type:
            continue
        label = type_labels.get(stype, stype)
        print(f"\n[{label}]")
        print(f"  {'Format':20s} {'Overall':>8s} {'Steps':>7s} {'Constr':>8s} {'Contract':>9s} {'n':>3s}")
        print(f"  {'-' * 60}")
        for f in by_type[stype]:
            print(
                f"  {f['format']:20s} {f['avg_overall']:>8.2f} "
                f"{f['avg_step_coverage']:>7.2f} {f['avg_constraint_respect']:>8.2f} "
                f"{f['avg_output_contract']:>9.2f} {f['count']:>3d}"
            )
else:
    print("No scenario_type data available in cached results.")

## Pairwise Wins

For each scenario, which format scored highest? Count wins across all scenarios. Ties (within 0.01) are counted separately.

## Key Findings (73 runs, 19 scenarios, claude-sonnet-4-6)

### The overall gap is modest (+4pp) — but it concentrates where it matters

LML Aggressive: 0.84 vs Raw Markdown: 0.80 overall. On standard/easy scenarios, all formats
perform equally (~0.95). The advantage emerges on **hard scenarios** (+11pp: 0.76 vs 0.65)
and **constraint resistance** (+18pp: 0.86 vs 0.68).

### Constraint resistance is the key differentiator

When the user pressures the model to skip steps ("just code it", "just approve it", "force
push NOW"), explicit `@red_flag` blocks in LML are harder to ignore than italic text in
Markdown. LML scores 0.86 vs Markdown's 0.68 on constraint resistance scenarios — the
largest gap in the benchmark.

### Pairwise wins tell a nuanced story

Of 19 scenarios: 11 ties, Raw Markdown wins 3, JSON IR wins 2, HTML wins 2, LML wins 1.
LML rarely wins outright — but when it does, the margin is large. Markdown wins are small
(~5pp). LML's strength is consistent performance with fewer catastrophic failures.

### Structure-per-token efficiency

LML Aggressive: 0.972 compliance per 1K tokens (best). Raw Markdown: 0.883. HTML: 0.662.
JSON IR: 0.211. LML achieves the highest compliance at the fewest tokens — the optimal
format for LLM consumption.

### Easy scenarios are not diagnostic

On easy/standard scenarios (SQL injection, eval detection), all formats score ~0.95. If your
use case is straightforward, format doesn't matter much. The value of typed blocks shows up
when tasks require judgment, restraint, or multi-step workflow compliance.

In [ ]:
def compute_pairwise_wins(results_list: list) -> dict:
    """For each scenario, which format scored highest?"""
    by_scenario = {}
    for r in results_list:
        name = r["scenario"]
        if name not in by_scenario:
            by_scenario[name] = []
        by_scenario[name].append(r)

    wins = {}
    ties = 0
    details = []
    for name, rs in sorted(by_scenario.items()):
        best = max(rs, key=lambda x: x["overall"])
        second = sorted(rs, key=lambda x: -x["overall"])[1] if len(rs) > 1 else best
        if best["overall"] - second["overall"] < 0.01:
            ties += 1
            details.append((name, "TIE", best["overall"]))
        else:
            winner = best["format"]
            wins[winner] = wins.get(winner, 0) + 1
            details.append((name, winner, best["overall"]))

    return {"wins": wins, "ties": ties, "total_scenarios": len(by_scenario), "details": details}


pw = compute_pairwise_wins(all_results)
print(f"Pairwise wins across {pw['total_scenarios']} scenarios:")
print(f"  Ties: {pw['ties']}")
for fmt, count in sorted(pw['wins'].items(), key=lambda x: -x[1]):
    print(f"  {fmt}: {count} wins")
print(f"\nDetails:")
for name, winner, score in pw['details']:
    print(f"  {name:50s} → {winner} ({score:.2f})")

# Save results
output = {
    "model_executor": MODEL_EXECUTOR,
    "model_judge": MODEL_JUDGE,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "scenario_count": len(SCENARIOS),
    "format_count": len(FORMATS),
    "total_runs": len(all_results),
    "format_summary": [dict(s) for s in compute_format_summary(all_results)],
    "token_efficiency": [dict(s) for s in compute_token_efficiency(all_results)],
    "difficulty_breakdown": {k: [dict(f) for f in v] for k, v in compute_difficulty_breakdown(all_results).items()},
    "scenario_type_breakdown": {k: [dict(f) for f in v] for k, v in compute_type_breakdown(all_results).items()},
    "pairwise_wins": pw,
    "scenarios": all_results,
}

output_path = Path(os.path.abspath("")) / "results.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved to {output_path}")
print(f"Total runs: {len(all_results)}")
print(f"Scenarios with results: {len(set(r['scenario'] for r in all_results))}")
if summaries:
    print(f"Best format: {summaries[0]['format']} (overall: {summaries[0]['avg_overall']:.2f})")